In [1]:
import coiled
import duckdb
import geopandas as gpd
import xarray as xr
import xarray_regrid  # noqa: F401  # side-effect import: registers .regrid namespace
import zarr

from srm import catalog

zarr.config.set({"async.concurrency": 64})

In [2]:
from srm import downscaling_utils, run_bcsd
from srm.downscaling_utils import interpolate_fine_to_coarse_grid
from srm.utils import Timer

In [3]:
cluster = coiled.Cluster(
    name="srm-fine-to-coarse",
    n_workers=[20, 40],
    region="us-west-2",
    worker_vm_types="m8g.large",
    scheduler_vm_types=["c8g.xlarge"],
    spot_policy="spot_with_fallback",
    tags={"Project": "SRM"},
)

client = cluster.get_client()
client

[2026-02-05 14:17:12,898][INFO    ][coiled] Fetching latest package priorities...
[2026-02-05 14:17:12,899][INFO    ][coiled.package_sync] Resolving your local /Users/clairezarakas/Documents/science/carbonplan-srm/uv.lock Python environment...
[2026-02-05 14:17:13,039][INFO    ][coiled.package_sync] Scanning 270 python packages...
[2026-02-05 14:17:13,347][INFO    ][coiled] Running pip check...
[2026-02-05 14:17:13,754][INFO    ][coiled] Validating environment...
[2026-02-05 14:17:15,052][INFO    ][coiled] Creating wheel for ~/Documents/science/carbonplan-srm/src...
[2026-02-05 14:17:15,138][INFO    ][coiled] Creating wheel for srm...
[2026-02-05 14:17:16,941][INFO    ][coiled] Creating wheel for xarray-regrid...
[2026-02-05 14:17:24,393][INFO    ][coiled] Uploading coiled_local_src...
[2026-02-05 14:17:25,626][INFO    ][coiled] Uploading srm...
[2026-02-05 14:17:26,554][INFO    ][coiled] Uploading xarray-regrid...
[2026-02-05 14:17:27,873][INFO    ][coiled] Creating software environme

<Client: 'tls://10.1.61.192:8786' processes=11 threads=22, memory=78.51 GiB>

# Compare two approaches

## Approach 1

In [4]:
#### Read in data
dict_all = run_bcsd.get_all_data(gcm="CESM2-WACCM", var_name="tas")

df = duckdb.sql(
    """install httpfs; load httpfs; install spatial; load spatial; SELECT name, ST_AsText(geom) as geometry FROM ST_Read('https://carbonplan-data.s3.us-west-2.amazonaws.com/countries-50m.json') WHERE NAME = 'South Africa'"""
).df()
df["geometry"] = gpd.GeoSeries.from_wkt(df["geometry"])
south_africa_geom = gpd.GeoDataFrame(df, geometry="geometry")

# Set bounds as a couple of degrees around South Africa
lon_min, lat_min, lon_max, lat_max = south_africa_geom.total_bounds

for key in dict_all:
    if key in ["obs"]:
        dict_all[key] = downscaling_utils.subset_space(
            dict_all[key],
            coord_bounds_list=[lat_min - 2, lat_max + 2, lon_min - 2, lon_max + 2],
        )

    else:
        dict_all[key] = downscaling_utils.subset_space(
            dict_all[key],
            coord_bounds_list=[lat_min - 1, lat_max + 1, lon_min - 1, lon_max + 1],
        )

##### Do regridding
with Timer("Interpolated obs to coarse grid", verbose=True):
    da_coarse_grid = dict_all["model_hist"].isel(time=0).drop_vars("time").reset_coords(drop=True)

    dict_all["obs_coarse"] = interpolate_fine_to_coarse_grid(
        da_fine_to_coarsen=dict_all["obs"], da_coarse_grid=da_coarse_grid
    )

Loaded data: 3.69 seconds
Interpolated obs to coarse grid: 155.82 seconds


## Approach 2 (from below) with spatial subsetting

In [5]:
with Timer("Interpolated obs to coarse grid", verbose=True):
    # coarse historical GCM grid - Note, single time slice, single var
    ds_coarse_grid = (
        catalog.get("CESM2-WACCM-Historical-icechunk").to_xarray()[["tasmax"]].isel(time=0)
    )
    ds_coarse_grid = downscaling_utils.subset_space(
        ds_coarse_grid,
        coord_bounds_list=[lat_min - 2, lat_max + 2, lon_min - 2, lon_max + 2],
    )

    # fine ERA5 - single var
    ds_fine_grid = catalog.get("ERA5").to_xarray()[["tasmax"]]
    ds_fine_grid = downscaling_utils.subset_space(
        ds_fine_grid,
        coord_bounds_list=[lat_min - 2, lat_max + 2, lon_min - 2, lon_max + 2],
    )

    # create a target grid from the GCM coarse dataset
    target_grid = ds_coarse_grid[["lat", "lon"]].drop_vars("time").reset_coords(drop=True)

    # use xarray regird
    ds_fine_regridded = ds_fine_grid.regrid.conservative(target_grid, latitude_coord="lat")
    ds_fine_regridded.to_zarr("s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-2", mode="w")

/Users/clairezarakas/Documents/science/carbonplan-srm/.venv/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


Interpolated obs to coarse grid: 216.98 seconds


## This shouls all be lazy and only take a few seconds

In [ ]:
# coarse historical GCM grid - Note, single time slice, single var
ds_coarse_grid = catalog.get("CESM2-WACCM-Historical-icechunk").to_xarray()[["tasmax"]].isel(time=0)

# fine ERA5 - single var
ds_fine_grid = catalog.get("ERA5").to_xarray()[["tasmax"]]

# create a target grid from the GCM coarse dataset
target_grid = ds_coarse_grid[["lat", "lon"]].drop_vars("time").reset_coords(drop=True)

# use xarray regird
ds_fine_regridded = ds_fine_grid.regrid.conservative(target_grid, latitude_coord="lat")

## Calling to_zarr or to_icechunk would trigger the computation. 

In [3]:
import time

In [ ]:
%%time
starttime = time.time()
# ~5 minutes
# We could speed this up with obstore + zarr backend

endtime = time.time()
print(endtime - starttime)

/opt/coiled/env/lib/python3.13/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


721.6458280086517
CPU times: user 10min 42s, sys: 1min 40s, total: 12min 22s
Wall time: 12min 1s


In [16]:
# shutdown our coiled cluster
client.shutdown()

2026-02-05 14:16:19,663 - distributed.deploy.adaptive - INFO - Adaptive scaling stopped: minimum=20 maximum=40. Reason: unknown
[2026-02-05 14:16:19,861][INFO    ][coiled] Cluster 1421646 deleted successfully.


In [ ]:
rtds = xr.open_zarr("s3://carbonplan-scratch/SRM-test/ERA5-coarse_test-2", chunks="auto")

In [ ]:
rtds.isel(time=0).sel(lat=slice(25, 50), lon=slice(-100, -70))["tasmax"].plot()

In [ ]:
ds_coarse_grid["tasmax"].sel(lat=slice(25, 50), lon=slice(-100, -70)).plot()

In [ ]:
ds_fine_grid.isel(time=0).sel(lat=slice(25, 50), lon=slice(-100, -70))["tasmax"].plot()